# DTA: drug/protein graph + drug/protein embedding

Model: drug GAT 8x64 -> GAT 4x64; protein GAT 8x64 -> GCN 256.
Four branches with max+mean pooling -> concatenation -> MLP 256->512->128->1. MSE loss, fixed LR.

Standalone notebook: add `train.pt` / `test.pt` in v4 format as inputs, select DATASET, then Run All.
Full checkpoints are saved every 100 epochs and at the final epoch; compatible checkpoints are found automatically.
`best_model.pt` is for inference only, not a full resume checkpoint.

Library cells are exported from `utils.py`, `models/gat_gcn.py`, and `training.py` using
`python notebook/sync_notebook.py`. Edit the repository model and regenerate to keep both versions aligned.


In [ ]:
from pathlib import Path

DATASET = 'davis'  # 'davis' or 'kiba'
DATA_DIR = None  # Set Path('/kaggle/input/...') if multiple datasets are available.
REPO_ROOT = next((p for base in (Path.cwd(), *Path.cwd().parents)
                  for p in (base, base / 'DATN_DTA')
                  if (p / 'training.py').is_file() and (p / 'models').is_dir()), Path.cwd())
if DATA_DIR is None:
    if Path('/kaggle/input').is_dir():
        candidates = sorted({p.parent for p in Path('/kaggle/input').rglob('train.pt')
                             if (p.parent / 'test.pt').is_file()})
        if len(candidates) != 1:
            raise ValueError(f'Found {len(candidates)} datasets; set DATA_DIR explicitly.')
        DATA_DIR = candidates[0]
    else:
        DATA_DIR = REPO_ROOT / 'data/four_source_embeddings' / DATASET
OUTPUT_DIR = (Path('/kaggle/working') if Path('/kaggle/working').is_dir()
              else REPO_ROOT / 'outputs') / f'{DATASET}_drug_gat8x64_gat4x64_dual_embedding_maxmean'
TRAIN_BATCH_SIZE = 512  # Global batch, split across GPUs.
TEST_BATCH_SIZE = 512
MAX_GPUS = 2  # Set to 1 to use a single GPU.
CUDA_DEVICE = 'cuda:0'
LR = 0.0005
NUM_EPOCHS = 1000  # Total epochs, including those completed before resuming.
CHECKPOINT_EVERY = 100
AUTO_RESUME = True
RESUME_PATH = None  # Full checkpoint Path; None scans output and /kaggle/input.
SEED = 42
SHUFFLE_TRAIN = False
LOG_INTERVAL = 20

MODEL_CONFIG = {'num_features_xd': 78, 'num_features_xt': 41, 'n_output': 1,
                'output_dim': 128, 'dropout': 0.2, 'protein_embedding_dim': 640}


In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec('torch_geometric') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'torch-geometric'])


In [ ]:
"""Four-source entity tables, indexed batches and portable training checkpoints.

This module does not import PyG, create a model, or run training on import.
Graph/embedding banks are read-only ordinary Python objects, not model buffers.
"""
from pathlib import Path
from types import SimpleNamespace
import csv
import hashlib
import json
import random

import numpy as np
import torch
import torch.nn as nn

class FourSourceSplit:
    """Load entities once; after GPU upload, retain indices, labels, and IDs for CSV export."""
    def __init__(self, path, expected_split, dataset=None, protein_embedding_dim=640):
        self.data = torch.load(path, map_location='cpu', weights_only=True, mmap=True)
        if self.data.get('format_version') != 4:
            raise ValueError(f'{path}: expected a v4 split file generated by create_data.py')
        self.dataset = self.data.get('dataset')
        if (self.dataset not in ('davis', 'kiba') or self.data.get('split') != expected_split
                or (dataset is not None and self.dataset != dataset)):
            raise ValueError(f'{path}: dataset/split does not match {dataset or "davis/kiba"} {expected_split}')
        self.ligand_ids = self.data['ligand_ids']
        self.protein_ids = self.data['protein_ids']
        self.pairs = self.data['pairs']
        self.labels = self.pairs['y'].float().reshape(-1)
        n = len(self.labels)
        if (self.pairs['y'].ndim != 1 or n == 0 or len(self.pairs['ligand_index']) != n
                or len(self.pairs['protein_index']) != n or len(self.pairs['row_id']) != n
                or not torch.isfinite(self.labels).all()):
            raise ValueError(f'{path}: invalid pair or label tables')
        nl, np_ = len(self.data['ligand_ids']), len(self.data['protein_ids'])
        if (len(self.data['ligand_graphs']) != nl or len(self.data['ligand_embeddings']) != nl
                or len(self.data['protein_graphs']) != np_
                or len(self.data['protein_embeddings']) != np_
                or len(self.data['protein_embedding_ids']) != np_
                or len(set(self.ligand_ids)) != nl or len(set(self.protein_ids)) != np_):
            raise ValueError(f'{path}: four-source table dimensions do not match')
        for key, size in [('ligand_index', nl), ('protein_index', np_)]:
            indices = self.pairs[key]
            if (indices.dtype != torch.long or indices.shape != (n,)
                    or indices.min() < 0 or indices.max() >= size):
                raise ValueError(f'{path}: invalid {key} indices')
        # Both sources contain unpadded token matrices; normalize dtype once per entity.
        for key, width in [('ligand_embeddings', 512),
                           ('protein_embeddings', protein_embedding_dim)]:
            for i, value in enumerate(self.data[key]):
                if (not isinstance(value, torch.Tensor) or value.ndim != 2
                        or value.shape[0] == 0 or value.shape[1] != width
                        or not value.is_floating_point() or not torch.isfinite(value).all()):
                    raise ValueError(f'{path}: {key}[{i}] must be finite with shape [L>0,{width}]')
                self.data[key][i] = value.detach().float()

    def __len__(self):
        return len(self.labels)

    def pair_ids(self, index):
        li = int(self.pairs['ligand_index'][index])
        pi = int(self.pairs['protein_index'][index])
        return self.ligand_ids[li], self.protein_ids[pi]


def pack_segments(values):
    """Concatenate segments once on CPU and store offsets without padding or pooling tokens."""
    lengths = torch.tensor([value.shape[0] for value in values], dtype=torch.long)
    ptr = torch.cat((torch.zeros(1, dtype=torch.long), lengths.cumsum(0)))
    return torch.cat(values, dim=0).contiguous(), ptr


def pack_graphs(graphs, width):
    for graph in graphs:
        x, edges = graph['x'], graph['edge_index']
        if (x.ndim != 2 or x.shape[0] == 0 or x.shape[1] != width
                or not torch.isfinite(x).all() or edges.ndim != 2 or edges.shape[0] != 2
                or edges.dtype != torch.long
                or (edges.numel() and (edges.min() < 0 or edges.max() >= len(x)))):
            raise ValueError(f'Graph requires finite x with shape [N>0,{width}] and valid edge_index')
    x, node_ptr = pack_segments([graph['x'].detach().float() for graph in graphs])
    # Edges store LOCAL node indices; add offsets only when selecting graphs for a batch.
    edges, edge_ptr = pack_segments([graph['edge_index'].t() for graph in graphs])
    return {'x': x, 'node_ptr': node_ptr, 'edges': edges, 'edge_ptr': edge_ptr}


def same_graph(a, b):
    return torch.equal(a['x'], b['x']) and torch.equal(a['edge_index'], b['edge_index'])


def build_cpu_tables(splits):
    """Merge train/test entities; use original graph IDs and sequence-hash protein embedding IDs."""
    if len({split.dataset for split in splits}) != 1:
        raise ValueError('DAVIS and KIBA cannot share one bank')
    ligands, proteins, protein_embeddings = {}, {}, {}
    for split in splits:
        data = split.data
        for i, lid in enumerate(split.ligand_ids):
            graph, embedding = data['ligand_graphs'][i], data['ligand_embeddings'][i]
            if lid in ligands:
                old_graph, old_embedding = ligands[lid]
                if not same_graph(old_graph, graph) or not torch.equal(old_embedding, embedding):
                    raise ValueError(f'Ligand {lid}: train/test data differ')
            else:
                ligands[lid] = (graph, embedding)
        for i, pid in enumerate(split.protein_ids):
            graph = data['protein_graphs'][i]
            eid = data['protein_embedding_ids'][i]
            embedding = data['protein_embeddings'][i]
            if pid in proteins:
                old_graph, old_eid = proteins[pid]
                if old_eid != eid or not same_graph(old_graph, graph):
                    raise ValueError(f'Protein {pid}: train/test data differ')
            else:
                proteins[pid] = (graph, eid)
            if eid in protein_embeddings:
                if not torch.equal(protein_embeddings[eid], embedding):
                    raise ValueError(f'Embedding {eid}: same ID has different values')
            else:
                protein_embeddings[eid] = embedding
    ligand_ids, protein_ids, embedding_ids = sorted(ligands), sorted(proteins), sorted(protein_embeddings)
    ligand_map = {key: i for i, key in enumerate(ligand_ids)}
    protein_map = {key: i for i, key in enumerate(protein_ids)}
    embedding_map = {key: i for i, key in enumerate(embedding_ids)}
    tables = {}
    for prefix, graphs, width in (
            ('ligand', [ligands[key][0] for key in ligand_ids], 78),
            ('protein', [proteins[key][0] for key in protein_ids], 41)):
        for key, value in pack_graphs(graphs, width).items():
            tables[f'{prefix}_{key}'] = value
    tables['ligand_embedding'], tables['ligand_embedding_ptr'] = pack_segments(
        [ligands[key][1] for key in ligand_ids])
    tables['protein_embedding'], tables['protein_embedding_ptr'] = pack_segments(
        [protein_embeddings[key] for key in embedding_ids])
    tables['protein_to_embedding'] = torch.tensor(
        [embedding_map[proteins[key][1]] for key in protein_ids], dtype=torch.long)
    return tables, ligand_map, protein_map


def execution_devices(device, use_data_parallel=True, max_gpus=2):
    """Put the primary GPU first; default to at most two GPUs, with single-GPU/CPU fallback."""
    device = torch.device(device)
    if max_gpus < 1:
        raise ValueError('MAX_GPUS must be >= 1')
    if device.type != 'cuda':
        return [device]
    count = torch.cuda.device_count()
    primary = device.index if device.index is not None else torch.cuda.current_device()
    if not 0 <= primary < count:
        raise ValueError(f'Primary GPU cuda:{primary} is not among the {count} available GPUs')
    ids = [primary]
    if use_data_parallel:
        ids += [i for i in range(count) if i != primary]
    return [torch.device('cuda', i) for i in ids[:max_gpus]]


def take_segments(values, ptr, indices):
    """Look up segments and build batch vectors on the bank's device (GPU during training)."""
    starts = ptr.index_select(0, indices)
    lengths = ptr.index_select(0, indices + 1) - starts
    batch_ptr = torch.cat((lengths.new_zeros(1), lengths.cumsum(0)))
    owners = torch.repeat_interleave(torch.arange(indices.numel(), device=indices.device), lengths)
    local = torch.arange(owners.numel(), device=indices.device) - batch_ptr.index_select(0, owners)
    positions = starts.index_select(0, owners) + local
    return values.index_select(0, positions), owners, batch_ptr


class DeviceFourSourceStore:
    """Each GPU holds a tensor dictionary; replicas share a read-only Python store."""
    def __init__(self, cpu_tables, devices, ligand_map, protein_map):
        self.devices = tuple(torch.device(device) for device in devices)
        if not self.devices or len(set(self.devices)) != len(self.devices):
            raise ValueError('Expected a non-empty list of distinct devices')
        self.device = self.devices[0]  # Pair indices and labels reside on the primary GPU.
        self.ligand_id_to_index = ligand_map
        self.protein_id_to_index = protein_map
        self.nbytes = sum(value.numel() * value.element_size() for value in cpu_tables.values())
        self.tables = {}
        for target in self.devices:
            if target.type == 'cuda':
                free, _ = torch.cuda.mem_get_info(target)
                if self.nbytes > free:
                    raise RuntimeError(f'Bank on {target} requires {self.nbytes / 1024**3:.3f} GiB, '
                                       f'but only {free / 1024**3:.3f} GiB is free. Restart the session to release the old bank.')
            # Upload once per GPU. The bank is not a model buffer, so DataParallel
            # does not broadcast/copy it on each forward pass, and checkpoints exclude it.
            self.tables[str(target)] = {
                key: value.detach().to(target) for key, value in cpu_tables.items()}
            print(f'Four sources on {target}: {self.nbytes / 1024**3:.3f} GiB; '
                  f'{len(ligand_map)} ligand graphs, {len(protein_map)} protein graphs, '
                  f'{cpu_tables["protein_embedding_ptr"].numel() - 1} protein embeddings')

    def on(self, device):
        key = str(torch.device(device))
        if key not in self.tables:
            raise ValueError(f'Bank is not prepared on {key}')
        return self.tables[key]

    def graph_batch(self, prefix, indices):
        t = self.on(indices.device)
        x, batch, node_ptr = take_segments(t[f'{prefix}_x'], t[f'{prefix}_node_ptr'], indices)
        edges, edge_batch, _ = take_segments(t[f'{prefix}_edges'], t[f'{prefix}_edge_ptr'], indices)
        edge_index = (edges + node_ptr.index_select(0, edge_batch).unsqueeze(1)).t().contiguous()
        return x, edge_index, batch

    def lookup(self, pair_indices):
        # DataParallel splits INDICES; each replica looks up its local batch on the same device.
        t = self.on(pair_indices.device)
        li, pi = pair_indices[:, 0], pair_indices[:, 1]
        drug_x, drug_edges, drug_batch = self.graph_batch('ligand', li)
        prot_x, prot_edges, prot_batch = self.graph_batch('protein', pi)
        drug_embedding, drug_embedding_batch, _ = take_segments(
            t['ligand_embedding'], t['ligand_embedding_ptr'], li)
        embedding_indices = t['protein_to_embedding'].index_select(0, pi)
        protein_embedding, protein_embedding_batch, _ = take_segments(
            t['protein_embedding'], t['protein_embedding_ptr'], embedding_indices)
        # Tensors already reside on the GPU; wrap attributes without CPU collation or copying.
        return (SimpleNamespace(x=drug_x, edge_index=drug_edges, batch=drug_batch,
                     drug_embedding=drug_embedding, drug_embedding_batch=drug_embedding_batch),
                SimpleNamespace(x=prot_x, edge_index=prot_edges, batch=prot_batch,
                     protein_embedding=protein_embedding, protein_embedding_batch=protein_embedding_batch))


class IndexBatches:
    """Yield only GPU pair indices and labels per batch; do not fetch graphs/tokens on CPU."""
    def __init__(self, split, store, batch_size, shuffle=False, seed=42):
        if batch_size < 1:
            raise ValueError('Batch size must be >= 1')
        self.size, self.batch_size = len(split), batch_size
        self.shuffle, self.device = shuffle, store.device
        self.generator = torch.Generator(device=self.device).manual_seed(seed)
        ligand_remap = torch.tensor([store.ligand_id_to_index[key] for key in split.ligand_ids])
        protein_remap = torch.tensor([store.protein_id_to_index[key] for key in split.protein_ids])
        self.pairs = torch.stack((
            ligand_remap.index_select(0, split.pairs['ligand_index']),
            protein_remap.index_select(0, split.pairs['protein_index'])), dim=1).to(self.device)
        self.labels = split.labels.reshape(-1, 1).to(self.device)

    def __len__(self):
        return (self.size + self.batch_size - 1) // self.batch_size

    def reset_seed(self, seed):
        self.generator.manual_seed(seed)

    def __iter__(self):
        order = (torch.randperm(self.size, generator=self.generator, device=self.device)
                 if self.shuffle else torch.arange(self.size, device=self.device))
        for start in range(0, self.size, self.batch_size):
            rows = order[start:start + self.batch_size]
            yield self.pairs.index_select(0, rows), self.labels.index_select(0, rows)


def unwrap_model(model):
    return model.module if isinstance(model, nn.DataParallel) else model

def save_checkpoint(checkpoint, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    try:
        torch.save(checkpoint, temporary)
        temporary.replace(path)
    finally:
        temporary.unlink(missing_ok=True)

def save_test_predictions(path, dataset, labels, predictions):
    if not len(dataset) == len(labels) == len(predictions):
        raise ValueError('Test sample and prediction counts do not match')
    with Path(path).open('w', newline='', encoding='utf-8') as handle:
        writer = csv.writer(handle)
        writer.writerow(['row_id', 'ligand_id', 'protein_id', 'affinity', 'predicted_affinity'])
        for i, (label, prediction) in enumerate(zip(labels, predictions)):
            ligand_id, protein_id = dataset.pair_ids(i)
            writer.writerow([dataset.pairs['row_id'][i], ligand_id, protein_id,
                             float(label), float(prediction)])

def cpu_snapshot(value):
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().clone()
    if isinstance(value, dict):
        return {key: cpu_snapshot(item) for key, item in value.items()}
    if isinstance(value, list):
        return [cpu_snapshot(item) for item in value]
    if isinstance(value, tuple):
        return tuple(cpu_snapshot(item) for item in value)
    return value

def dataset_signature(splits):
    # Validate pair/label order and IDs without rereading large embedding tensors.
    digest = hashlib.sha256()
    for split in splits:
        digest.update(json.dumps([split.ligand_ids, split.protein_ids],
                                 ensure_ascii=False).encode('utf-8'))
        for key in ('ligand_index', 'protein_index', 'y'):
            value = split.pairs[key].detach().cpu().contiguous()
            digest.update(str((key, tuple(value.shape), str(value.dtype))).encode())
            digest.update(value.numpy().tobytes())
    return digest.hexdigest()

def model_signature(model):
    core = unwrap_model(model)
    description = {
        'modules': [(name, ('__main__' if name == '' else type(module).__module__), type(module).__qualname__, repr(module))
                    for name, module in core.named_modules()],
        'state': [(key, list(value.shape), str(value.dtype))
                  for key, value in core.state_dict().items()],
    }
    return hashlib.sha256(json.dumps(description, sort_keys=True).encode()).hexdigest()

def capture_rng(batches, devices):
    numpy_state = np.random.get_state()
    return {
        'python': random.getstate(),
        'numpy': {'name': numpy_state[0], 'keys': numpy_state[1].tolist(),
                  'position': numpy_state[2], 'has_gauss': numpy_state[3],
                  'cached_gaussian': numpy_state[4]},
        'torch_cpu': torch.get_rng_state().clone(),
        'torch_cuda': [torch.cuda.get_rng_state(target).cpu()
                       for target in devices if target.type == 'cuda'],
        'batch_generator': batches.generator.get_state().cpu(),
        'batch_generator_device': str(batches.generator.device),
    }

def restore_rng(state, batches, devices):
    random.setstate(state['python'])
    ns = state['numpy']
    np.random.set_state((ns['name'], np.array(ns['keys'], dtype=np.uint32),
                         ns['position'], ns['has_gauss'], ns['cached_gaussian']))
    torch.set_rng_state(state['torch_cpu'])
    targets = [target for target in devices if target.type == 'cuda']
    if len(targets) == len(state['torch_cuda']):
        for target, value in zip(targets, state['torch_cuda']):
            torch.cuda.set_rng_state(value, target)
    else:
        print('GPU count changed: keeping the new session CUDA seed; exact replay is unavailable.')
    saved_type = torch.device(state['batch_generator_device']).type
    if saved_type == torch.device(batches.generator.device).type:
        batches.generator.set_state(state['batch_generator'])
    else:
        print('CPU/CUDA generator changed: keeping the new session shuffle seed.')

def resume_problem(payload, expected):
    if not isinstance(payload, dict) or payload.get('checkpoint_kind') != 'training_resume_v1':
        return 'not a full resume checkpoint (old best/weights file lacks optimizer/RNG state)'
    required = ('model_state_dict', 'optimizer_state_dict', 'epoch', 'best_checkpoint',
                'history', 'rng_state', 'best_predictions')
    if any(key not in payload for key in required):
        return 'missing training state'
    for key, value in expected.items():
        if payload.get(key) != value:
            return f'{key} differs from the current run'
    epoch = payload['epoch']
    if not isinstance(epoch, int) or epoch < 1:
        return 'invalid epoch'
    if [row.get('epoch') for row in payload['history']] != list(range(1, epoch + 1)):
        return 'history does not match epoch'
    best = payload['best_checkpoint']
    score_key = 'validation_mse' if isinstance(best, dict) and best.get('selection_split') == 'validation' else 'test_mse'
    if (not isinstance(best, dict) or 'model_state_dict' not in best
            or not 1 <= best.get('epoch', 0) <= epoch
            or not np.isfinite(best.get(score_key, float('nan')))):
        return 'invalid best checkpoint'
    return None

def find_resume_checkpoint(roots, expected, explicit_path=None):
    if explicit_path is not None:
        candidates = [Path(explicit_path)]
        if not candidates[0].is_file():
            raise FileNotFoundError(f'RESUME_PATH not found: {explicit_path}')
    else:
        candidates = set()
        for root in roots:
            root = Path(root)
            if root.is_dir():
                for pattern in ('*checkpoint*.pt', '*checkpoint*.pth', 'best_model.pt',
                                'best_until_epoch_*.pt'):
                    candidates.update(root.rglob(pattern))
        candidates = sorted(candidates, key=lambda p: str(p))
    selected, selected_path = None, None
    for path in candidates:
        try:
            # New resume files use zip-based torch.save; mmap does not upload weights to the GPU.
            payload = torch.load(path, map_location='cpu', weights_only=True, mmap=True)
            problem = resume_problem(payload, expected)
        except Exception as exc:
            payload, problem = None, f'cannot read: {exc}'
        if problem:
            if explicit_path is not None:
                raise ValueError(f'Cannot resume {path}: {problem}')
            print(f'Skipping {path.name}: {problem}')
            continue
        if selected is None or payload['epoch'] > selected['epoch']:
            selected, selected_path = payload, path
    return selected_path, selected

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_data(data_dir, dataset, device, batch_size=512, test_batch_size=512,
              max_gpus=2, shuffle=False, seed=42):
    """Validate splits, upload one bank per device, then release CPU entity lists."""
    data_dir = Path(data_dir)
    train_data = FourSourceSplit(data_dir / 'train.pt', 'train', dataset)
    test_data = FourSourceSplit(data_dir / 'test.pt', 'test', dataset)
    cpu_tables, ligand_map, protein_map = build_cpu_tables((train_data, test_data))
    devices = execution_devices(device, use_data_parallel=max_gpus > 1, max_gpus=max_gpus)
    store = DeviceFourSourceStore(cpu_tables, devices, ligand_map, protein_map)
    train_batches = IndexBatches(train_data, store, batch_size, shuffle, seed)
    test_batches = IndexBatches(test_data, store, test_batch_size, False, seed)
    train_data.data = test_data.data = None
    print(f'{dataset.upper()}: {len(train_data):,} train / {len(test_data):,} test')
    print('Devices:', [str(d) for d in devices], '| global batch:', batch_size)
    return train_data, test_data, store, train_batches, test_batches


In [ ]:
"""Four-source DTA model; matches the current original-copy notebook."""
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv, GCNConv
from torch_geometric.nn import global_mean_pool as gap
from torch_geometric.nn import global_max_pool as gmp
# GAT  model
class GATnGCN_Model(torch.nn.Module):
    def __init__(self, data_store, num_features_xd=78, n_output=1, num_features_xt=41,
                      output_dim=128, dropout=0.2, protein_embedding_dim=640):
        super(GATnGCN_Model, self).__init__()
        # The store is a plain Python object; state_dict contains only model weights.
        self.data_store = data_store
        if output_dim != 128:
            raise ValueError('Fusion requires output_dim=128')
        # Ligand graph: GAT 8x64 = 512 -> GAT 4x64 = 256; max+mean pooling = 512.
        self.conv1 = GATConv(num_features_xd, 64, heads=8, concat=True)
        self.conv2 = GATConv(512, 64, heads=4, concat=True)
        self.fc_g1 = torch.nn.Linear(512, 256)
        self.fc_g2 = torch.nn.Linear(256, output_dim)

        # Replace ECFP with Chemformer ligand embeddings. Pool only in forward.
        self.fc_drug_embedding = nn.Linear(2 * 512, 128)
        self.bn_drug_embedding = nn.BatchNorm1d(128)
        # Protein graph: GAT 8x64 -> GCN 256, with 41 input features and separate weights.
        self.conv_prot1 = GATConv(num_features_xt, 64, heads=8, concat=True)
        self.conv_prot2 = GCNConv(512, 256)
        self.fc_prot_g1 = nn.Linear(512, 256)
        self.fc_prot_g2 = nn.Linear(256, output_dim)

        # Protein ESM-2: concatenate max/mean pooling, then project 1280 -> 128.
        self.fc_protein_embedding = nn.Linear(2 * protein_embedding_dim, 128)
        self.bn_protein_embedding = nn.BatchNorm1d(128)

        #concat
        self.fc_xd = nn.Linear(256,128)
        self.fc_xp = nn.Linear(256,128)

        # Small prediction head: 256 -> 512 -> 128 -> affinity.
        self.fc1 = nn.Linear(256, 512)
        self.fc2 = nn.Linear(512, 128)
        self.out = nn.Linear(128, n_output)

        # activation and regularization
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)

    def forward(self, pair_indices):
        # Look up all four sources on the GPU at the start of each forward pass.
        data_drug, data_protein = self.data_store.lookup(pair_indices)
        # graph input feed-forward
        x_drug, edge_index_drug, batch_drug = data_drug.x, data_drug.edge_index, data_drug.batch

        x_drug = self.conv1(x_drug, edge_index_drug)
        x_drug = self.relu(x_drug)
        x_drug = self.conv2(x_drug, edge_index_drug)
        x_drug = self.relu(x_drug)
        # apply global max pooling (gmp) and global mean pooling (gap)
        x_drug = torch.cat([gmp(x_drug, batch_drug), gap(x_drug, batch_drug)], dim=1)
        x_drug = self.relu(self.fc_g1(x_drug))
        x_drug = self.dropout(x_drug)
        x_drug = self.fc_g2(x_drug)

        # Tokens are concatenated; drug_embedding_batch identifies their samples.
        # The bank preserves raw tokens; max+mean pooling happens only here.
        # Concatenate [max, mean], pooling each sample separately: [B,1024].
        x_drug_embedding = torch.cat([
            gmp(data_drug.drug_embedding, data_drug.drug_embedding_batch),
            gap(data_drug.drug_embedding, data_drug.drug_embedding_batch),
        ], dim=1)
        x_drug_embedding = self.fc_drug_embedding(x_drug_embedding)
        # BatchNorm needs >=2 training samples; use running statistics for a singleton batch.
        if self.training and x_drug_embedding.size(0) == 1:
            x_drug_embedding = F.batch_norm(
                x_drug_embedding, self.bn_drug_embedding.running_mean,
                self.bn_drug_embedding.running_var, self.bn_drug_embedding.weight,
                self.bn_drug_embedding.bias, training=False, eps=self.bn_drug_embedding.eps)
        else:
            x_drug_embedding = self.bn_drug_embedding(x_drug_embedding)
        x_drug_embedding = self.relu(x_drug_embedding)
        x_drug_embedding = self.dropout(x_drug_embedding)

        # protein input feed-forward:
        x_prots, edge_index_prots, prots_batch= data_protein.x, data_protein.edge_index, data_protein.batch
        # Protein: GAT -> GCN -> max+mean -> Linear/ReLU/Dropout -> Linear.
        x_prots = self.conv_prot1(x_prots, edge_index_prots)
        x_prots = self.relu(x_prots)
        x_prots = self.conv_prot2(x_prots, edge_index_prots)
        x_prots = self.relu(x_prots)
        x_prots = torch.cat([gmp(x_prots, prots_batch), gap(x_prots, prots_batch)], dim=1)
        x_prots = self.relu(self.fc_prot_g1(x_prots))
        x_prots = self.dropout(x_prots)
        x_prots = self.fc_prot_g2(x_prots)
        # Pool tokens per sample using protein_embedding_batch, not the graph node batch.
        # Both pooling operations use each sample's actual tokens without padding.
        # Concatenation order: [max (640), mean (640)] -> [B,1280].
        x_protein_embedding = torch.cat([
            gmp(data_protein.protein_embedding, data_protein.protein_embedding_batch),
            gap(data_protein.protein_embedding, data_protein.protein_embedding_batch),
        ], dim=1)
        x_protein_embedding = self.fc_protein_embedding(x_protein_embedding)
        if self.training and x_protein_embedding.size(0) == 1:
            x_protein_embedding = F.batch_norm(
                x_protein_embedding, self.bn_protein_embedding.running_mean,
                self.bn_protein_embedding.running_var, self.bn_protein_embedding.weight,
                self.bn_protein_embedding.bias, training=False, eps=self.bn_protein_embedding.eps)
        else:
            x_protein_embedding = self.bn_protein_embedding(x_protein_embedding)
        x_protein_embedding = self.relu(x_protein_embedding)
        x_protein_embedding = self.dropout(x_protein_embedding)

        # Ablation: concatenate graph features directly, without cross-attention or residuals.
        xd = torch.cat((x_drug, x_drug_embedding), dim=1)       # [B,256]
        xp = torch.cat((x_prots, x_protein_embedding), dim=1)  # [B,256]
        xd = self.relu(self.fc_xd(xd))
        xp = self.relu(self.fc_xp(xp))

        xc = torch.cat((xd,xp), 1)
        # Apply ReLU/Dropout after each hidden layer; the output layer is linear.
        xc = self.fc1(xc)
        xc = self.relu(xc)
        xc = self.dropout(xc)
        xc = self.fc2(xc)
        xc = self.relu(xc)
        xc = self.dropout(xc)
        out = self.out(xc)
        return out

In [ ]:
"""Train the four-source DTA model with fixed LR and resumable checkpoints."""
from __future__ import annotations

import argparse
import csv
from dataclasses import dataclass, field
from pathlib import Path
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


ROOT = REPO_ROOT
MODEL_TYPE = 'GATnGCN_DrugGAT8x64_GAT4x64_ProteinGATGCN_DualEmbeddingMaxMean_SmallMLP'
DEFAULT_MODEL_CONFIG = {'num_features_xd': 78, 'num_features_xt': 41, 'n_output': 1,
                        'output_dim': 128, 'dropout': 0.2, 'protein_embedding_dim': 640}


@dataclass
class TrainingConfig:
    dataset: str = 'davis'
    data_dir: Path | None = None
    output_dir: Path | None = None
    device: str = 'cuda:0'
    max_gpus: int = 2
    batch_size: int = 512
    test_batch_size: int = 512
    lr: float = 0.0005
    epochs: int = 1000
    checkpoint_every: int = 100
    auto_resume: bool = True
    resume_path: Path | None = None
    input_dir: Path = Path('/kaggle/input')
    shuffle: bool = False
    seed: int = 42
    log_interval: int = 20
    model_config: dict = field(default_factory=lambda: dict(DEFAULT_MODEL_CONFIG))

    def __post_init__(self):
        if self.dataset not in ('davis', 'kiba'):
            raise ValueError('dataset must be davis or kiba')
        if min(self.batch_size, self.test_batch_size, self.max_gpus, self.epochs,
               self.checkpoint_every) < 1 or self.lr <= 0 or not np.isfinite(self.lr):
            raise ValueError('Batch sizes, epochs, max_gpus, checkpoint_every and LR must be positive')
        if self.log_interval < 0:
            raise ValueError('log_interval must be >= 0')
        self.data_dir = Path(self.data_dir) if self.data_dir else ROOT / 'data/four_source_embeddings' / self.dataset
        base = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else ROOT / 'outputs'
        self.output_dir = Path(self.output_dir) if self.output_dir else base / f'{self.dataset}_drug_gat8x64_gat4x64_dual_embedding_maxmean'
        self.input_dir = Path(self.input_dir)
        self.resume_path = Path(self.resume_path) if self.resume_path else None


def configure_model(model_config, store):

    for target in store.devices:
        store.on(target)
    core = GATnGCN_Model(data_store=store, **model_config).to(store.device)
    if len(store.devices) > 1:
        gpu_ids = [target.index for target in store.devices]
        return nn.DataParallel(core, device_ids=gpu_ids, output_device=gpu_ids[0], dim=0)
    return core

def train(model, batches, optimizer, epoch, log_interval=20):
    model.train()
    total_loss, count = 0.0, 0
    for batch_idx, (pair_indices, labels) in enumerate(batches):
        optimizer.zero_grad(set_to_none=True)
        output = model(pair_indices)
        # Optimize MSE over the full batch gathered from all GPUs.
        loss = F.mse_loss(output, labels)
        value = loss.item()
        if not np.isfinite(value):
            raise FloatingPointError(f'Loss contains NaN/Inf at epoch {epoch}, batch {batch_idx + 1}')
        loss.backward()
        optimizer.step()
        size = pair_indices.size(0)
        total_loss += value * size
        count += size
        if log_interval and (batch_idx == 0 or (batch_idx + 1) % log_interval == 0):
            print(f'Epoch {epoch} [{count}/{batches.size}] MSE: {value:.6f}')
    return total_loss / count

@torch.no_grad()
def predicting(model, batches):
    model.eval()
    labels, predictions = [], []
    for pair_indices, batch_labels in batches:
        output = model(pair_indices)
        predictions.append(output.cpu().reshape(-1))
        labels.append(batch_labels.cpu().reshape(-1))
    return torch.cat(labels).numpy(), torch.cat(predictions).numpy()

def run_training(config: TrainingConfig):
    """Run up to the total epoch target; importing this module never starts a run."""
    seed_everything(config.seed)
    device = torch.device(config.device if torch.cuda.is_available() else 'cpu')
    train_data, test_data, data_store, train_batches, test_batches = load_data(
        config.data_dir, config.dataset, device, config.batch_size, config.test_batch_size,
        config.max_gpus, config.shuffle, config.seed)
    # Match notebook initialization after bank creation.
    seed_everything(config.seed)
    train_batches.reset_seed(config.seed)
    model = configure_model(config.model_config, data_store)
    optimizer = torch.optim.Adam(model.parameters(), lr=config.lr)
    config.output_dir.mkdir(parents=True, exist_ok=True)
    run_identity = {
        'model_config': dict(config.model_config),
        'model_signature': model_signature(model),
        'dataset_signature': dataset_signature((train_data, test_data)),
        'train_config': {'optimizer': 'Adam', 'lr': config.lr, 'batch_size': config.batch_size,
                         'test_batch_size': config.test_batch_size, 'shuffle': config.shuffle,
                         'seed': config.seed, 'loss': 'mse'},
    }
    best_mse, best_epoch, best_checkpoint = float('inf'), -1, None
    best_predictions, history, start_epoch = None, [], 1
    resume_path, resume = (find_resume_checkpoint(
        [config.output_dir, config.input_dir], run_identity, config.resume_path)
        if config.auto_resume or config.resume_path is not None else (None, None))
    if resume is not None:
        unwrap_model(model).load_state_dict(resume['model_state_dict'], strict=True)
        # PyTorch load_state_dict places optimizer state on each parameter's device.
        optimizer.load_state_dict(resume['optimizer_state_dict'])
        best_checkpoint = cpu_snapshot(resume['best_checkpoint'])
        best_mse, best_epoch = best_checkpoint['test_mse'], best_checkpoint['epoch']
        best_predictions = cpu_snapshot(resume['best_predictions'])
        history = list(resume['history'])
        start_epoch = resume['epoch'] + 1
        # Restore best-model artifacts even if this run produces no further improvement.
        save_checkpoint(best_checkpoint, config.output_dir / 'best_model.pt')
        save_test_predictions(config.output_dir / 'best_test_predictions.csv', test_data,
                              test_data.labels.cpu().numpy(), best_predictions.numpy())
        restore_rng(resume['rng_state'], train_batches, data_store.devices)
        print(f'Resuming from {resume_path}: completed epoch {start_epoch - 1}; '
              f'continuing at {start_epoch}, best MSE={best_mse:.6f} at epoch {best_epoch}')
        del resume
    else:
        print('No compatible resume checkpoint or config.auto_resume=False: training from scratch.')
    print(f'Trainable parameters: {sum(p.numel() for p in model.parameters()):,}')
    print('DataParallel:', isinstance(model, nn.DataParallel), '| devices:',
          [str(target) for target in data_store.devices])
    if start_epoch > config.epochs:
        print(f'Checkpoint reached epoch {start_epoch - 1}; target is {config.epochs}, no further training.')

    # Preserve the legacy checkpoint-selection rule based on test MSE.
    # Test data participates in model selection and is not an independent evaluation set.
    with (config.output_dir / 'history.csv').open('w', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(handle, fieldnames=['epoch', 'train_mse', 'test_mse', 'seconds'])
        writer.writeheader()
        writer.writerows(history)
        handle.flush()
        for epoch in range(start_epoch, config.epochs + 1):
            started = time.perf_counter()
            train_mse = train(model, train_batches, optimizer, epoch, config.log_interval)
            labels, predictions = predicting(model, test_batches)
            if not np.isfinite(predictions).all():
                raise FloatingPointError('Test predictions contain NaN/Inf')
            test_mse = float(np.mean((labels.astype(np.float64) - predictions.astype(np.float64)) ** 2))
            if test_mse < best_mse:
                best_mse, best_epoch = test_mse, epoch
                best_checkpoint = {
                    'model_type': MODEL_TYPE,
                    'model_config': dict(config.model_config),
                    'model_signature': run_identity['model_signature'],
                    'model_state_dict': cpu_snapshot(unwrap_model(model).state_dict()),
                    'epoch': epoch, 'test_mse': test_mse, 'selection_split': 'test',
                    'dataset': config.dataset, 'dataset_format_version': 4, 'loss_type': 'mse',
                    'fusion_head': 'dense_mlp_256_512_128',
                    'pair_fusion': 'concat_ligand_protein',
                    'ligand_graph_encoder': 'gat_8x64_gat_4x64_maxmean_fc256_out128',
                    'protein_graph_encoder': 'gat_8x64_gcn512to256_maxmean_fc256_out128',
                    'data_loading': 'per_device_bank_indexed_in_forward',
                    'attention_fusion': 'direct_graph_embedding_concat',
                    'ligand_embedding_pooling': 'concat_max_mean_over_tokens_in_model',
                    'protein_embedding_pooling': 'concat_max_mean_over_tokens_in_model',
                }
                best_predictions = torch.from_numpy(predictions.copy()).cpu()
                save_checkpoint(best_checkpoint, config.output_dir / 'best_model.pt')
                save_test_predictions(config.output_dir / 'best_test_predictions.csv', test_data, labels, predictions)
            record = {'epoch': epoch, 'train_mse': train_mse, 'test_mse': test_mse,
                      'seconds': time.perf_counter() - started}
            history.append(record)
            writer.writerow(record)
            handle.flush()
            # Save the CURRENT model and optimizer state, with a separate best checkpoint.
            if epoch % config.checkpoint_every == 0 or epoch == config.epochs:
                payload = {
                    'checkpoint_kind': 'training_resume_v1', **run_identity,
                    'epoch': epoch,
                    'model_state_dict': cpu_snapshot(unwrap_model(model).state_dict()),
                    'optimizer_state_dict': cpu_snapshot(optimizer.state_dict()),
                    'best_checkpoint': best_checkpoint,
                    'best_predictions': best_predictions,
                    'history': list(history),
                    'rng_state': capture_rng(train_batches, data_store.devices),
                }
                path = config.output_dir / f'training_checkpoint_epoch_{epoch:04d}.pt'
                save_checkpoint(payload, path)
                del payload
                print('Saved resume checkpoint:', path)
            print(f'Epoch {epoch}: train MSE={train_mse:.6f}, test MSE={test_mse:.6f} '
                  f'| best={best_mse:.6f} at epoch {best_epoch}')

    if best_checkpoint is not None:
        # The resume checkpoint retains the final state; inference uses the best model.
        unwrap_model(model).load_state_dict(best_checkpoint['model_state_dict'])
        model.eval()
        print('Best model:', config.output_dir / 'best_model.pt')

    return {'model': model, 'best_epoch': best_epoch, 'best_mse': best_mse,
            'history': history, 'output_dir': config.output_dir, 'test_batches': test_batches}


def build_parser():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--dataset', choices=['davis', 'kiba'], default='davis')
    parser.add_argument('--data-dir', type=Path, help='Directory containing format-v4 train.pt/test.pt')
    parser.add_argument('--output-dir', type=Path)
    parser.add_argument('--device', default='cuda:0')
    parser.add_argument('--max-gpus', type=int, default=2, help='1 disables DataParallel')
    parser.add_argument('--batch-size', type=int, default=512, help='GLOBAL batch across GPUs')
    parser.add_argument('--test-batch-size', type=int, default=512)
    parser.add_argument('--lr', type=float, default=0.0005, help='Fixed LR, no scheduler')
    parser.add_argument('--epochs', type=int, default=1000, help='TOTAL epochs, including resumed history')
    parser.add_argument('--checkpoint-every', type=int, default=100)
    parser.add_argument('--auto-resume', action=argparse.BooleanOptionalAction, default=True)
    parser.add_argument('--resume-path', type=Path)
    parser.add_argument('--input-dir', type=Path, default=Path('/kaggle/input'))
    parser.add_argument('--shuffle', action=argparse.BooleanOptionalAction, default=False)
    parser.add_argument('--seed', type=int, default=42)
    parser.add_argument('--log-interval', type=int, default=20)
    return parser


def main(argv=None):
    config = TrainingConfig(**vars(build_parser().parse_args(argv)))
    return run_training(config)




In [ ]:
config = TrainingConfig(
    dataset=DATASET, data_dir=DATA_DIR, output_dir=OUTPUT_DIR,
    device=CUDA_DEVICE, max_gpus=MAX_GPUS,
    batch_size=TRAIN_BATCH_SIZE, test_batch_size=TEST_BATCH_SIZE,
    lr=LR, epochs=NUM_EPOCHS, checkpoint_every=CHECKPOINT_EVERY,
    auto_resume=AUTO_RESUME, resume_path=RESUME_PATH,
    seed=SEED, shuffle=SHUFFLE_TRAIN, log_interval=LOG_INTERVAL,
    model_config=MODEL_CONFIG)
result = run_training(config)
print('Best epoch:', result['best_epoch'], '| test MSE:', result['best_mse'])


In [ ]:
# run_training restores the best checkpoint before returning the model.
model = result['model']
pair_indices, _ = next(iter(result['test_batches']))
with torch.no_grad():
    example_predictions = model(pair_indices)
print('Example predictions:', example_predictions[:5].cpu().flatten())
del pair_indices, example_predictions


Outputs: `best_model.pt`, `best_test_predictions.csv`, `history.csv`, and
`training_checkpoint_epoch_XXXX.pt`. Add a full checkpoint as input in the next session to resume.
Checkpoints retain current weights and optimizer state, a separate best model, history, and RNG state.
Changes to architecture, split, LR, or batch size prevent automatic resume from an old checkpoint.
Moving code from a notebook to a module does not itself break checkpoint compatibility for the same architecture.
Test MSE selects the best checkpoint in this legacy workflow, so the test set is not an independent evaluation set.
Save `/kaggle/working` files as Kaggle outputs to reuse them in another session.
